# Inference Tokens — Chart Generation

Reproducible generation of the Inference Tokens visual evidence charts.

**Currently produces:** `P-01` and `P-03`

| | |
|---|---|
| **Source data** | `company_disclosures.csv`, exported from the Dataset Register workbook (`03_Company_Disclosures`) |
| **Outputs** | `charts/P-01.svg`, `charts/P-01.png`, `charts/P-03.svg`, `charts/P-03.png` |
| **Published to** | `inference-tokens/charts/` in the `ai-data-room` repository |

### Rules this notebook follows

1. Every plotted value comes from `company_disclosures.csv`. Nothing is invented, estimated or filled.
2. The disclosed value is never overwritten. Unit conversion produces a **new** number; the original stays visible in the source file and in the chart labels.
3. Figures with materially different scopes are not pooled into one comparison without annotation.
4. Plot IDs are stable and match the Excel Plot Index and the public gallery.

### Normalisation basis

As stated in the methodology:

```
monthly equivalent = tokens/minute × 1,440 × 30.44
annual  equivalent = tokens/minute × 60 × 24 × 365
```

The 30.44-day month is the mean calendar month. Both conversions assume the disclosed rate held constant for the period — an assumption, not a measurement, and one that is stated on the face of every chart that relies on it.

## 1 · Setup

Set `REPO` to a local clone of `ai-data-room`, or leave it as-is to work in a scratch directory and download the charts at the end.

In [ ]:
from pathlib import Path

REPO = Path.cwd()                      # a clone of ai-data-room, or scratch
DATA = REPO / "inference-tokens" / "data" / "company_disclosures.csv"
CHARTS = REPO / "inference-tokens" / "charts"
CHARTS.mkdir(parents=True, exist_ok=True)

print("repo    :", REPO)
print("data    :", DATA, "—", "found" if DATA.exists() else "NOT FOUND (see next cell)")
print("charts  :", CHARTS)

### If the data file is not present

Upload `company_disclosures.csv` when prompted. It must contain these columns:

`Company · Disclosure_date · Metric · Value_as_disclosed · Unit_as_disclosed · Normalized_value · Normalized_unit · Scope · Product_or_surface · Disclosure_context · Source_name · Source_link · Source_type · Methodology_reference · Plot_ID · Notes`

In [ ]:
if not DATA.exists():
    try:
        from google.colab import files          # Colab upload prompt
        up = files.upload()
        DATA.parent.mkdir(parents=True, exist_ok=True)
        name = next(iter(up))
        DATA.write_bytes(up[name])
        print("saved to", DATA)
    except ImportError:
        raise SystemExit(
            f"Place company_disclosures.csv at {DATA} before continuing."
        )

## 2 · Load and inspect the disclosures

Read the file and look at what is actually available before plotting anything.

In [ ]:
import pandas as pd

df = pd.read_csv(DATA)
print(f"{len(df)} disclosures across {df['Company'].nunique()} organisations\n")

cols = ["Company", "Disclosure_date", "Value_as_disclosed", "Unit_as_disclosed", "Scope"]
rate = df[df["Unit_as_disclosed"].str.contains("tokens/minute", na=False)]
print("Rate-based throughput disclosures:")
display(rate[cols].reset_index(drop=True))

## 3 · Conversion helpers

`parse_rate` pulls the numeric magnitude out of a disclosed string such as `"more than 15 billion"` **without** discarding the qualifier — the qualifier is reported back so the caller knows the figure is a floor rather than a point estimate.

In [ ]:
import re

MIN_TO_MONTH = 1440 * 30.44
MIN_TO_YEAR = 60 * 24 * 365
SCALE = {"thousand": 1e3, "million": 1e6, "billion": 1e9,
         "trillion": 1e12, "quadrillion": 1e15}


def parse_disclosed(text):
    """'more than 15 billion' -> (15e9, 'more than'). Qualifier is preserved."""
    t = str(text).strip().lower()
    qualifier = ""
    for q in ("more than", "over", "nearly", "approximately", "crossed", "surpassed"):
        if t.startswith(q):
            qualifier, t = q, t[len(q):].strip()
            break
    m = re.match(r"([\d.,]+)\s*(\w+)?", t)
    if not m:
        raise ValueError(f"cannot parse disclosed value: {text!r}")
    value = float(m.group(1).replace(",", "")) * SCALE.get(m.group(2) or "", 1.0)
    return value, qualifier


def rate_to_year(tokens_per_min):
    return tokens_per_min * MIN_TO_YEAR


def rate_to_month(tokens_per_min):
    return tokens_per_min * MIN_TO_MONTH


# check the helpers reproduce the normalised values already recorded in the workbook
g22, q = parse_disclosed("22 billion")
print(f"22 bn/min -> {rate_to_month(g22)/1e12:,.1f} T/month   (workbook states 964.3)")
print(f"22 bn/min -> {rate_to_year(g22)/1e15:,.2f} quadrillion/yr (workbook states 11.56)")
o15, q = parse_disclosed("more than 15 billion")
print(f"15 bn/min -> {rate_to_year(o15)/1e15:,.2f} quadrillion/yr (workbook states 7.88), "
      f"qualifier retained: {q!r}")

## 4 · House style

One style block, applied to every chart, so the gallery stays visually consistent as it grows.

In [ ]:
import textwrap
import matplotlib.pyplot as plt

NAVY, INK, MUTED, RULE = "#1f3864", "#1a1a1a", "#6b7280", "#d7dbe2"
SERIES = {"current": "#1f3864", "prior": "#9aa9c4",
          "other": "#6b8f71", "scope": "#b4763a"}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "text.color": INK, "axes.labelcolor": INK, "axes.edgecolor": RULE,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "svg.fonttype": "none",   # keeps SVG text as text
    "figure.dpi": 100, "svg.hashsalt": "inference-tokens",
})


def frame(fig, ax, plot_id, title, subtitle, source, methodology, note):
    fig.text(0.035, 0.972, plot_id, ha="left", va="top", fontsize=10.5,
             fontweight="bold", color=NAVY, family="monospace")
    fig.text(0.035, 0.928, title, ha="left", va="top", fontsize=15.5,
             fontweight="bold", color=INK)
    for i, line in enumerate(textwrap.wrap(subtitle, 122)):
        fig.text(0.035, 0.882 - i * 0.030, line, ha="left", va="top",
                 fontsize=10.2, color=MUTED)
    note_lines = textwrap.wrap(note, 133) if note else []
    y = 0.052 + 0.026 * (2 + len(note_lines))
    fig.lines.append(plt.Line2D([0.035, 0.965], [y, y], transform=fig.transFigure,
                                color=RULE, linewidth=0.8))
    fig.text(0.035, y - 0.022, f"Source: {source}", fontsize=8.6, color=MUTED,
             ha="left", va="top")
    fig.text(0.035, y - 0.048, f"Methodology: {methodology}", fontsize=8.6,
             color=MUTED, ha="left", va="top")
    for i, line in enumerate(note_lines):
        fig.text(0.035, y - 0.074 - i * 0.024, ("Note: " if i == 0 else "      ") + line,
                 fontsize=8.4, color=MUTED, style="italic", ha="left", va="top")
    ax.grid(axis="y", color=RULE, linewidth=0.7, alpha=0.9)
    ax.set_axisbelow(True)


def save(fig, plot_id):
    for ext, kw in (("svg", {}), ("png", {"dpi": 200})):
        p = CHARTS / f"{plot_id}.{ext}"
        fig.savefig(p, format=ext, facecolor="white", **kw)
        print("wrote", p)

## 5 · P-01 — Annualised inference throughput by disclosing provider

Three providers, two measurement bases. Google and OpenAI publish an instantaneous tokens-per-minute rate; Microsoft publishes a completed fiscal-year total. The bases are coloured differently rather than pooled, and the chart states plainly that the bars must not be summed — Azure AI Foundry is a gateway, so its volume is attributed to the model owner under the de-duplication rule and overlaps with OpenAI's.

In [ ]:
def pick(company, starts, unit_contains):
    hit = df[(df["Company"] == company)
             & (df["Value_as_disclosed"].str.startswith(starts))
             & (df["Unit_as_disclosed"].str.contains(unit_contains, regex=False))]
    if hit.empty:
        raise LookupError(f"no disclosure for {company} / {starts}")
    return hit.iloc[0]


g_prev = pick("Alphabet / Google", "16 billion", "tokens/minute")
g_now = pick("Alphabet / Google", "22 billion", "tokens/minute")
o_prev = pick("OpenAI", "6 billion", "tokens/minute")
o_now = pick("OpenAI", "more than 15 billion", "tokens/minute")
ms = pick("Microsoft", "over 500 trillion", "FY2025")

Q = 1e15
bars = [
    ("Google\nmodel APIs",
     ("Q1 2026\n16 bn/min", rate_to_year(parse_disclosed(g_prev["Value_as_disclosed"])[0]) / Q),
     ("Q2 2026\n22 bn/min", rate_to_year(parse_disclosed(g_now["Value_as_disclosed"])[0]) / Q),
     "rate"),
    ("OpenAI\nAPIs",
     ("Oct 2025\n6 bn/min", rate_to_year(parse_disclosed(o_prev["Value_as_disclosed"])[0]) / Q),
     ("Mar 2026\n15 bn/min", rate_to_year(parse_disclosed(o_now["Value_as_disclosed"])[0]) / Q),
     "rate"),
    ("Microsoft\nAzure AI Foundry",
     None,
     ("FY2025 total\n500 T tokens", parse_disclosed(ms["Value_as_disclosed"])[0] / Q),
     "period"),
]

fig = plt.figure(figsize=(11.0, 7.4))
ax = fig.add_axes([0.085, 0.315, 0.88, 0.50])
w = 0.34
for i, (prov, prior, current, kind) in enumerate(bars):
    if prior:
        ax.bar(i - w / 2, prior[1], w, color=SERIES["prior"], edgecolor="white", linewidth=0.8)
        ax.text(i - w / 2, prior[1] + 0.22, f"{prior[1]:.2f}", ha="center", va="bottom",
                fontsize=9, color=MUTED)
        ax.text(i - w / 2, -0.40, prior[0], ha="center", va="top", fontsize=7.6, color=MUTED)
        cx = i + w / 2
    else:
        cx = i
    ax.bar(cx, current[1], w, color=SERIES["current" if kind == "rate" else "other"],
           edgecolor="white", linewidth=0.8)
    ax.text(cx, current[1] + 0.22, f"{current[1]:.2f}", ha="center", va="bottom",
            fontsize=9.5, fontweight="bold", color=INK)
    ax.text(cx, -0.40, current[0], ha="center", va="top", fontsize=7.6, color=MUTED)

ax.set_xticks(range(len(bars)))
ax.set_xticklabels([b[0] for b in bars], fontsize=10.5)
ax.tick_params(axis="x", length=0, pad=34)
ax.set_ylabel("Annualised tokens per year (quadrillions)", fontsize=10)
ax.set_ylim(0, 13.6); ax.set_xlim(-0.7, 2.7)
ax.legend([plt.Rectangle((0, 0), 1, 1, color=SERIES[k]) for k in ("prior", "current", "other")],
          ["Earlier disclosure (rate, annualised)", "Latest disclosure (rate, annualised)",
           "Completed fiscal year (no annualisation)"],
          loc="upper right", frameon=False, fontsize=8.8)

frame(fig, ax, "P-01",
      "Annualised inference throughput by disclosing provider",
      "Token throughput as disclosed by each provider, converted to a common annual basis. "
      "Google and OpenAI publish an instantaneous rate; Microsoft publishes a completed "
      "fiscal-year total.",
      "Alphabet Q2 2026 earnings; OpenAI announcement; Microsoft FY2025 Q4 earnings",
      "\u00a77.3 \u2014 Method 1, Direct Provider Throughput",
      "Scopes differ and the bars must not be summed. Microsoft's figure covers Azure AI "
      "Foundry, a gateway: models served through it are counted at the model owner "
      "(\u00a77.9, rule 1), so it overlaps with OpenAI's. Rate-based figures are spot rates "
      "annualised, which assumes the rate held for a full year. Microsoft's figure is "
      "FY2025, ended 30 June 2025.")
save(fig, "P-01")
plt.show()

## 6 · P-03 — Google API throughput sits inside the all-surfaces figure

The chart whose only job is to stop a double-count. Google publishes two token figures with different scopes, and the larger already contains the smaller. Drawn as nested volumes, never as two additive bars.

In [ ]:
api = pick("Alphabet / Google", "22 billion", "tokens/minute")
allsurf = pick("Alphabet / Google", "over 3.2 quadrillion", "tokens/month")

api_month = rate_to_month(parse_disclosed(api["Value_as_disclosed"])[0]) / 1e12   # T/month
all_month = parse_disclosed(allsurf["Value_as_disclosed"])[0] / 1e12              # T/month
print(f"model APIs      {api_month:,.0f} T/month")
print(f"all surfaces    {all_month:,.0f} T/month")

fig = plt.figure(figsize=(11.0, 7.0))
ax = fig.add_axes([0.085, 0.30, 0.88, 0.50])
ax.barh(0.6, all_month, height=0.42, color=SERIES["scope"], alpha=0.28,
        edgecolor=SERIES["scope"], linewidth=1.4)
ax.barh(0.6, api_month, height=0.42, color=SERIES["current"], edgecolor="white", linewidth=1.0)
ax.text(api_month / 2, 0.66, "model APIs only", ha="center", va="center",
        color="white", fontsize=10, fontweight="bold")
ax.text(api_month / 2, 0.54, f"{api_month:,.0f} T/month", ha="center", va="center",
        color="white", fontsize=10.5)
ax.text(all_month, 0.90, f"all Google surfaces  \u2014  {all_month:,.0f} T/month",
        ha="right", va="center", fontsize=10.5, color=SERIES["scope"], fontweight="bold")
ax.annotate("", xy=(api_month, 0.28), xytext=(all_month, 0.28),
            arrowprops=dict(arrowstyle="<->", color=MUTED, linewidth=1.0))
ax.text((api_month + all_month) / 2, 0.20,
        f"consumer and enterprise surfaces not in the API figure \u2014 "
        f"{all_month - api_month:,.0f} T/month",
        ha="center", va="top", fontsize=8.8, color=MUTED)
ax.set_xlim(0, all_month * 1.06); ax.set_ylim(0, 1.35); ax.set_yticks([])
ax.set_xlabel("Tokens per month (trillions)", fontsize=10)
ax.spines["left"].set_visible(False)
ax.grid(axis="x", color=RULE, linewidth=0.7)

frame(fig, ax, "P-03",
      "Google API throughput sits inside the all-surfaces figure",
      "Two Google disclosures on a common monthly basis. The all-surfaces figure already "
      "contains the model-API figure, so the two describe nested volumes rather than "
      "separate ones.",
      "Alphabet Q2 2026 earnings; Google I/O 2026 keynote",
      "\u00a77.9 \u2014 De-duplication rule 2; \u00a72.6 \u2014 Global share",
      "These two figures must never be added together. The API figure is a spot rate of 22 "
      "billion tokens/minute (Q2 2026) put on a 30.44-day month; the all-surfaces figure is "
      "disclosed directly as over 3.2 quadrillion tokens/month (May 2026). The dates differ "
      "and the all-surfaces figure spans multimodal and consumer traffic, so the gap shown "
      "is indicative of scope, not a precise residual.")
save(fig, "P-03")
plt.show()

## 7 · Publish

Commit the regenerated charts and rebuild the gallery. The gallery HTML is generated from the Plot Index, so a new chart file appears on the site automatically — no HTML editing.

```bash
python build/build_site.py
git add inference-tokens/charts inference-tokens/index.html
git commit -m "Regenerate P-01 and P-03 from company disclosures"
git push
```

In Colab, download the outputs instead:

In [ ]:
try:
    from google.colab import files
    for p in sorted(CHARTS.glob("P-*")):
        files.download(str(p))
except ImportError:
    print("Not running in Colab. Charts are in:", CHARTS)
    for p in sorted(CHARTS.glob("P-*")):
        print(" ", p.name)

---

## Adding the next chart

1. Find the Plot ID in `02_Plot_Index` and read its `What_the_chart_shows` field — it states the required axes and the annotation that must appear on the chart.
2. Confirm the source dataset is available. If it is not, stop: request the file rather than substituting another dataset.
3. Write a cell that loads the real data, reusing `frame()` and `save()` so the chart matches the house style.
4. Save as `P-NN.svg` and `P-NN.png`.
5. Run `build/build_site.py`, commit, push. The chart replaces its Pending placeholder in the gallery.

### Datasets still needed for the remaining charts

The company disclosures in this notebook cover the provider-throughput charts. Most other sections need files that are staged in the project Drive rather than in this repository — the OpenRouter rankings exports, the Azure inference traces, the Epoch AI chip and data-centre tables, the MLPerf benchmark export and the pricing registries. Each chart's required dataset is named in the `Dataset` field of the Plot Index.